In [ ]:
from sbi_particle_physics.managers.backup import Backup
from sbi_particle_physics.config import DATA_DIR, ACCEPTANCE_COEFFS_PATH, C9, DEFAULT_STRIDE, DEFAULT_PRE_N, DEFAULT_PRERUNS, REAL_DATA
from sbi_particle_physics.objects.model import Model
from sbi_particle_physics.managers.imperfections_diagnostics import ImperfectionsDiagnostics
from sbi_particle_physics.managers.model_diagnostics import ModelDiagnostics
from sbi_particle_physics.managers.real_data import RealData
import torch

In [ ]:
device = "cpu" # this small test works on cpu
max_files = 3000
n_points = 10000
files = Backup.detect_files(DATA_DIR / "data_6") # data_7
print(f"#files found {len(files)}")
files = files[:max_files]
raw_data, raw_parameters, _ = Backup.load_data(files, device)
print(raw_data.shape)

In [ ]:
real_raw_data, _ = RealData.load_n_points(REAL_DATA, n_points=n_points, device=device)
print(real_raw_data[:,:].mean(dim=0))
print(real_raw_data[:,-1].mean())

In [ ]:
def select_closest_c9_dataset(raw_data, c9_values, C9_true):
    distances = torch.abs(c9_values - C9_true)
    idx = torch.argmin(distances).item()
    x_sim_selected = raw_data[idx]
    print(f"Selected C9 = {c9_values[idx].item():.4f} (need to be close to {C9})")
    return x_sim_selected

In [ ]:
x_real = real_raw_data # Observed real
x_sim = select_closest_c9_dataset(raw_data, raw_parameters, 3.34) # "Full" (ie include imperfections)
x_obs = raw_data[1] # Observed but simulated
x_sim_long = raw_data[2:]

In [ ]:
def apply_mask(x, q2_max=6.0, mB_max=5.4):
    mask = (x[:, 0] <= q2_max) & (x[:, -1] <= mB_max)
    return x[mask]

#x_real = apply_mask(x_real)
#x_sim = apply_mask(x_sim)
#x_obs = apply_mask(x_obs)
#print(f"shape x _long before masking: {x_sim_long.shape}")
#x_sim_long = [apply_mask(x) for x in x_sim_long]
#print(f"shape x _long after masking: {[x.shape for x in x_sim_long]}")

#print(x_real.shape, x_sim.shape, x_obs.shape)

In [ ]:
def mask_columns_with_zero(x, columns_to_keep):
    y = torch.zeros_like(x)
    y[:, columns_to_keep] = x[:, columns_to_keep]
    return y

cols_to_keep = [0,1,2,3,4] # 1 ok

#x_real = mask_columns_with_zero(x_real, cols_to_keep)
#x_sim = mask_columns_with_zero(x_sim, cols_to_keep)
#x_obs = mask_columns_with_zero(x_obs, cols_to_keep)
#x_sim_long = [mask_columns_with_zero(x, cols_to_keep) for x in x_sim_long]

In [ ]:
ImperfectionsDiagnostics.compare_real_vs_simulated(x_real, x_sim, bins=40)

In [ ]:
ImperfectionsDiagnostics.chi2_test(x_sim, x_real, bins=40)

In [ ]:
ModelDiagnostics.misspecification_test(x_sim_long, x_o=x_real) # <----

In [ ]:
ModelDiagnostics.misspecification_test_mmd(x_sim_long, x_o=x_real)
# only needs to be between 0.2->0.8 (model is wrong if <0.05)

In [ ]:
ImperfectionsDiagnostics.chi2_test(x_sim, x_obs, bins=40)

In [ ]:
ModelDiagnostics.misspecification_test(x_sim_long, x_o=x_obs) # <----

In [ ]:
ModelDiagnostics.misspecification_test_mmd(x_sim_long, x_o=x_obs)
# only needs to be between 0.2->0.8 (model is wrong if <0.05)